In [18]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import numpy as np, pandas as pd, nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ciabd12/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/ciabd12/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [19]:
df = pd.read_csv('Reviews.csv')

# Binarizar: 1-2 estrellas = Negativo, 4-5 = Positivo (eliminar 3)
df = df[df['Score'] != 3]
df['Sentimiento'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

# Usar una muestra manejable (el dataset es enorme)
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

textos  = df['Text'].tolist()
etiquetas = np.array(df['Sentimiento'].tolist())

In [20]:
vocab_size   = 5000   
max_length   = 100    
trunc_type   = 'post'
padding_type = 'post'
oov_tok      = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(textos)

sequences = tokenizer.texts_to_sequences(textos)
padded    = pad_sequences(sequences, maxlen=max_length,
                          padding=padding_type, truncating=trunc_type)

# Train/Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    padded, etiquetas, test_size=0.2, random_state=42)

In [21]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 16, input_length=max_length),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam',
              metrics=['accuracy'])
model.summary()

/home/ciabd12/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(X_train, y_train,
                    epochs=10,           
                    validation_data=(X_test, y_test),
                    batch_size=64)

loss, acc = model.evaluate(X_test, y_test)
print(f"Precisión en test: {acc:.2%}")

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.8480 - loss: 0.4909 - val_accuracy: 0.8450 - val_loss: 0.4310
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8500 - loss: 0.4182 - val_accuracy: 0.8450 - val_loss: 0.4219
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.8500 - loss: 0.3718 - val_accuracy: 0.8470 - val_loss: 0.3688
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8955 - loss: 0.2563 - val_accuracy: 0.8670 - val_loss: 0.3790
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9390 - loss: 0.1635 - val_accuracy: 0.8640 - val_loss: 0.3576
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9625 - loss: 0.1050 - val_accuracy: 0.8730 - val_loss: 0.4289
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9815 - loss: 0.0610 - val_accuracy: 0.8590 - val_loss: 0.4231
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9885 - loss: 0.0397 - val_accuracy: 0.8660 - v

In [23]:
def predecir_sentimiento(frase):
    seq = tokenizer.texts_to_sequences([frase])
    pad = pad_sequences(seq, maxlen=max_length, padding=padding_type)
    pred = model.predict(pad, verbose=0)[0][0]
    return 1 if pred > 0.5 else 0

df['Prediccion'] = [predecir_sentimiento(t) for t in textos]

resenas_positivas = df[df['Prediccion'] == 1]['Text'].tolist()
resenas_negativas = df[df['Prediccion'] == 0]['Text'].tolist()

print(f"Positivas: {len(resenas_positivas)} ({len(resenas_positivas)/len(df):.1%})")
print(f"Negativas: {len(resenas_negativas)} ({len(resenas_negativas)/len(df):.1%})")

Positivas: 4354 (87.1%)
Negativas: 646 (12.9%)


In [24]:
def ideas_clave(lista_resenas, n=5):
    texto_unido = " ".join(lista_resenas)
    stop_en = stopwords.words('english')   # Dataset en inglés
    
    vectorizer = TfidfVectorizer(stop_words=stop_en, max_features=200)
    tfidf = vectorizer.fit_transform([texto_unido])
    
    palabras    = vectorizer.get_feature_names_out()
    puntuaciones = tfidf.toarray()[0]
    
    top_idx = puntuaciones.argsort()[-n:][::-1]
    return [palabras[i] for i in top_idx]

puntos_fuertes = ideas_clave(resenas_positivas, n=5)
puntos_debiles = ideas_clave(resenas_negativas, n=5)

print("Puntos fuertes:", puntos_fuertes)
print("Puntos débiles:", puntos_debiles)

Puntos fuertes: ['br', 'like', 'good', 'great', 'one']
Puntos débiles: ['br', 'like', 'product', 'taste', 'would']


In [26]:
print("=" * 50)
print("      INFORME DE AUDITORÍA DE SATISFACCIÓN")
print("=" * 50)
print(f"Producto: Amazon Fine Food (muestra de {len(df)} reseñas)")
print(f"Reseñas Positivas: {len(resenas_positivas)} ({len(resenas_positivas)/len(df):.1%})")
print(f"Reseñas Negativas: {len(resenas_negativas)} ({len(resenas_negativas)/len(df):.1%})")
print(f"\nPUNTOS FUERTES: {puntos_fuertes}")
print(f"PUNTOS DÉBILES: {puntos_debiles}")

      INFORME DE AUDITORÍA DE SATISFACCIÓN
Producto: Amazon Fine Food (muestra de 5000 reseñas)
Reseñas Positivas: 4354 (87.1%)
Reseñas Negativas: 646 (12.9%)

PUNTOS FUERTES: ['br', 'like', 'good', 'great', 'one']
PUNTOS DÉBILES: ['br', 'like', 'product', 'taste', 'would']
